### Cargar los datos
Cargar la base principal desde el archivo Parquet y el diccionario de variables desde el archivo CSV.

In [4]:
import pandas as pd

df = pd.read_parquet("data/raw/stuff_model_df.parquet")
dic = pd.read_csv("data/raw/stuff_plus_data_dictionary.csv")

df.shape

(635002, 84)

In [5]:
df.columns.tolist()

['year',
 'PitchUID',
 'game_anon_id',
 'pitcher_anon_id',
 'batter_anon_id',
 'catcher_anon_id',
 'PitcherThrows',
 'BatterSide',
 'altitude_category',
 'AutoPitchType',
 'RelSpeed',
 'EffectiveVelo',
 'ZoneSpeed',
 'SpinRate',
 'SpinAxis',
 'Tilt',
 'RelHeight',
 'RelSide',
 'Extension',
 'VertBreak',
 'InducedVertBreak',
 'HorzBreak',
 'VertRelAngle',
 'HorzRelAngle',
 'VertApprAngle',
 'HorzApprAngle',
 'SpeedDrop',
 'ZoneTime',
 'x0',
 'y0',
 'z0',
 'vx0',
 'vy0',
 'vz0',
 'ax0',
 'ay0',
 'az0',
 'pfxx',
 'pfxz',
 'PitchTrajectoryXc0',
 'PitchTrajectoryXc1',
 'PitchTrajectoryXc2',
 'PitchTrajectoryYc0',
 'PitchTrajectoryYc1',
 'PitchTrajectoryYc2',
 'PitchTrajectoryZc0',
 'PitchTrajectoryZc1',
 'PitchTrajectoryZc2',
 'PitchCall',
 'KorBB',
 'play_result',
 'hit_type',
 'ExitSpeed',
 'Angle',
 'Direction',
 'Distance',
 'is_swing',
 'is_whiff',
 'is_contact',
 'is_called_strike',
 'is_swinging_strike',
 'is_ball_in_play',
 'is_batted',
 'is_hit',
 'single',
 'double',
 'triple',
 '

In [6]:
dic.head(30)

,column_name,definition,data_type,unit_or_values,category,feature_role,use_as_model_feature,notes
0,year,Season year extracted from the local pitch date.,integer,YYYY,season/context,context_only,"Yes, for season normalization or splits only",Derived from Trackman Date. Useful for train/t...
1,PitchUID,Unique pitch identifier for the pitch event.,string,GUID / unique ID,identifier,id_only,No,Use only to return predictions and join result...
2,game_anon_id,Anonymous game identifier created by Diablos f...,string,game_######,anonymized identifier,grouping_only,No,"Use for grouping, validation splits, or aggreg..."
3,pitcher_anon_id,Anonymous pitcher identifier created by Diablo...,string,pitcher_#####,anonymized identifier,grouping_only,No,Use for aggregation and returning pitcher-leve...
4,batter_anon_id,Anonymous batter identifier created by Diablos...,string,batter_#####,anonymized identifier,grouping_only,No,Use for aggregation or handedness checks only....
5,catcher_anon_id,Anonymous catcher identifier created by Diablo...,string,catcher_#####,anonymized identifier,grouping_only,No,Use for grouping or diagnostics only. Avoid as...
6,PitcherThrows,Throwing hand of the pitcher for the pitch.,categorical,"Right, Left",pitcher context,stuff_feature,Yes,Useful because pitch movement and approach are...
7,BatterSide,Hitting side of the batter for the pitch.,categorical,"Right, Left",batter context,stuff_feature,Yes,Useful for platoon/context modeling.
8,altitude_category,Ballpark altitude bucket/category assigned in ...,categorical,Dataset-specific altitude buckets,park/environment context,context_only,Yes,NaN
9,AutoPitchType,Pitch type automatically classified by Trackma...,categorical,"Four-Seam, Sinker, Cutter, Changeup, Splitter,...",pitch classification,stuff_feature,Yes,NaN


In [7]:
dic.columns.tolist()

['column_name',
 'definition',
 'data_type',
 'unit_or_values',
 'category',
 'feature_role',
 'use_as_model_feature',
 'notes']

### Buscar variables de resultado
Filtramos el diccionario de datos para encontrar columnas relacionadas con resultados del lanzamiento, swing, contacto, bateo o carreras. Eso me ayuda a identificar variables útiles para medir el desempeño.

In [8]:
dic[dic["category"].astype(str).str.contains(
    "outcome|result|contact|batted|swing|event|run",
    case=False,
    na=False
)][["column_name", "definition", "unit_or_values"]]


,column_name,definition,unit_or_values
48,PitchCall,"Pitch-level result call such as called strike,...","StrikeCalled, StrikeSwinging, BallCalled, Ball..."
49,KorBB,Plate-appearance terminal strikeout/walk flag ...,"Strikeout, Walk, Undefined"
50,play_result,Cleaned/normalized ball-in-play or plate-appea...,Dataset-specific result labels
51,hit_type,Cleaned/normalized batted-ball type from the D...,"Ground ball, line drive, fly ball, popup, bunt..."
52,ExitSpeed,Speed of the batted ball off the bat.,mph
53,Angle,Vertical launch angle of the batted ball off t...,degrees
54,Direction,Horizontal launch direction of the batted ball...,degrees
55,Distance,Estimated carry distance of the batted ball.,feet
56,is_swing,Indicator that the batter swung at the pitch.,0/1 or TRUE/FALSE
57,is_whiff,Indicator that the batter swung and missed.,0/1 or TRUE/FALSE


### Revisar categorías de altitud
Cuenta cuántos lanzamientos hay en cada categoría de altitud. `dropna=False` hace que también se muestren los valores faltantes

In [9]:
df["altitude_category"].value_counts(dropna=False)

altitude_category
No Altitude         293875
Extreme Altitude    183267
Medium Altitude     154840
None                  3020
Name: count, dtype: int64

### Revisar tipos de lanzamiento
Cuenta cuántos lanzamientos hay de cada tipo de pitch. Sirve para saber qué tipos aparecen en la base y cuáles tienen suficientes observaciones para analizarlos.

In [10]:
df["AutoPitchType"].value_counts(dropna=False)

AutoPitchType
Four-Seam          217618
Slider             139129
Sinker             110897
Changeup            78925
Curveball           54024
Cutter              26468
Splitter             7289
Sweeper               588
Other                  61
OneSeamFastBall         1
TwoSeamFastBall         1
Knuckleball             1
Name: count, dtype: int64

# H1. La baja densidad reduce el movimiento inducido por spin

### Objetivo

Revisar si los lanzamientos realizados en condiciones de mayor altitud presentan menor movimiento inducido por spin.

`altitude_category` se utiliza para distinguir entre los diferentes entornos de altitud. 

Para medir el movimiento consideramos:

- `InducedVertBreak`: movimiento vertical inducido, en pulgadas.
- `HorzBreak`: movimiento horizontal, en pulgadas.

Como el movimiento depende fuertemente del tipo de lanzamiento, primero haremos el análisis de forma general y después separando por `AutoPitchType`.

In [11]:
h1 = df[
    [
        "altitude_category",
        "AutoPitchType",
        "SpinRate",
        "InducedVertBreak",
        "HorzBreak",
        "RelSpeed",
        "pitcher_anon_id"
    ]
].copy()

h1.shape

(635002, 7)

In [12]:
h1.isna().sum()

altitude_category    3020
AutoPitchType           0
SpinRate               80
InducedVertBreak      702
HorzBreak             702
RelSpeed                0
pitcher_anon_id      3272
dtype: int64

### Preparación de H1

Para revisar H1, nos quedamos solo con los datos que sí tienen:
- categoría de altitud
- movimiento vertical inducido
- movimiento horizontal

Se elimina la categoría `None` porque no corresponde a un nivel de altitud útil para comparar.

In [13]:
h1_clean = h1.dropna(
    subset=["altitude_category", "InducedVertBreak", "HorzBreak"]
).copy()

h1_clean = h1_clean[
    h1_clean["altitude_category"].isin(
        ["No Altitude", "Medium Altitude", "Extreme Altitude"]
    )
]

h1_clean.shape

(631280, 7)

### Comparación general por altitud

Comparar el movimiento promedio en los tres niveles de altitud para ver si hay algún patrón general

In [14]:
resumen_general = (
    h1_clean
    .groupby("altitude_category")
    [["InducedVertBreak", "HorzBreak"]]
    .mean()
    .round(2)
)

resumen_general

,InducedVertBreak,HorzBreak
altitude_category,,
Extreme Altitude,6.87,2.26
Medium Altitude,7.47,2.06
No Altitude,9.01,3.63


## Comparación por tipo de lanzamiento

Como cada tipo de pitch tiene un movimiento distinto, hay que comparar el movimiento promedio dentro de cada tipo de lanzamiento para evitar que la mezcla de pitches distorsione el resultado.

In [15]:
comparacion_tipo = (
    h1_clean
    .groupby(
        ["AutoPitchType", "altitude_category"],
        observed=True
    )
    [["InducedVertBreak", "HorzBreak"]]
    .mean()
    .round(2)
)

comparacion_tipo

InducedVertBreak  HorzBreak
AutoPitchType   altitude_category                             
Changeup        Extreme Altitude               5.20       4.18
                Medium Altitude                5.90       3.22
                No Altitude                    7.29       5.59
Curveball       Extreme Altitude              -6.36      -3.09
                Medium Altitude               -7.08      -3.41
                No Altitude                   -8.03      -3.07
Cutter          Extreme Altitude               8.21      -0.60
                Medium Altitude                9.37      -0.94
                No Altitude                   10.81      -0.58
Four-Seam       Extreme Altitude              13.07       4.36
                Medium Altitude               14.22       4.48
                No Altitude                   17.11       5.25
Other           Extreme Altitude              -7.06      -5.16
                Medium Altitude               -9.12      -9.58
                No Altitude                  -11.18     -11.99
Sinker          Extreme Altitude               7.98       6.01
                Medium Altitude                8.81       5.71
                No Altitude                   10.04       8.96
Slider          Extreme Altitude               2.34      -1.98
                Medium Altitude                2.57      -2.01
                No Altitude                    2.90      -1.97
Splitter        Extreme Altitude               2.44       4.65
                Medium Altitude                2.19       4.76
                No Altitude                    5.30       6.16
OneSeamFastBall Extreme Altitude              -0.34      11.11
Sweeper         Extreme Altitude              -2.05      -4.57
TwoSeamFastBall Extreme Altitude               5.60      12.97
Knuckleball     Extreme Altitude               2.59      -9.76

## Magnitud del movimiento

Como el movimiento horizontal y algunos movimientos verticales pueden tomar valores positivos o negativos según la dirección, comparar únicamente sus promedios puede ser engañoso.

Por eso también hay que comparar la magnitud del movimiento, usando el valor absoluto de cada componente.

In [16]:
h1_clean["AbsIVB"] = h1_clean["InducedVertBreak"].abs()
h1_clean["AbsHB"] = h1_clean["HorzBreak"].abs()

In [17]:
tipos_principales = [
    "Four-Seam",
    "Slider",
    "Sinker",
    "Changeup",
    "Curveball",
    "Cutter",
    "Splitter"
]

h1_principal = h1_clean[
    h1_clean["AutoPitchType"].isin(tipos_principales)
]

In [18]:
comparacion_magnitud = (
    h1_principal
    .groupby(
        ["AutoPitchType", "altitude_category"],
        observed=True
    )
    .agg(
        n=("AbsIVB", "size"),
        IVB_abs_promedio=("AbsIVB", "mean"),
        HB_abs_promedio=("AbsHB", "mean")
    )
    .round(2)
)

comparacion_magnitud


n  IVB_abs_promedio  HB_abs_promedio
AutoPitchType altitude_category                                          
Changeup      Extreme Altitude   19254              5.66             9.47
              Medium Altitude    17642              6.25            10.15
              No Altitude        41322              7.75            13.46
Curveball     Extreme Altitude   14386              6.68             7.17
              Medium Altitude    14063              7.31             7.54
              No Altitude        25094              8.55             9.48
Cutter        Extreme Altitude   10098              8.22             2.42
              Medium Altitude     6600              9.37             2.72
              No Altitude         9616             10.81             2.79
Four-Seam     Extreme Altitude   61792             13.07             7.22
              Medium Altitude    55329             14.22             7.56
              No Altitude        99489             17.11             9.44
Sinker        Extreme Altitude   31680              8.05            11.77
              Medium Altitude    23573              8.89            12.51
              No Altitude        55027             10.34            15.59
Slider        Extreme Altitude   42564              3.59             4.04
              Medium Altitude    36125              3.80             4.51
              No Altitude        59730              4.25             5.74
Splitter      Extreme Altitude    2203              3.60             5.03
              Medium Altitude     1507              3.64             5.25
              No Altitude         3540              7.10             7.45

## Magnitud total del movimiento

Para resumir el movimiento vertical y horizontal en una sola medida, hay que calcular la magnitud total del movimiento usando ambas componentes.

Después comparamos cuánto cambia entre `No Altitude` y `Extreme Altitude` para cada tipo de lanzamiento.

In [19]:
h1_clean["movement_magnitude"] = (
    h1_clean["InducedVertBreak"]**2 +
    h1_clean["HorzBreak"]**2
) ** 0.5

In [20]:
movimiento_total = (
    h1_clean[
        h1_clean["AutoPitchType"].isin(tipos_principales)
    ]
    .groupby(
        ["AutoPitchType", "altitude_category"],
        observed=True
    )
    ["movement_magnitude"]
    .mean()
    .unstack()
    .round(2)
)

movimiento_total

altitude_category,Extreme Altitude,Medium Altitude,No Altitude
AutoPitchType,,,
Changeup,11.64,12.55,16.28
Curveball,10.34,11.19,13.58
Cutter,8.78,9.97,11.39
Four-Seam,15.19,16.38,19.85
Sinker,14.53,15.62,19.12
Slider,6.07,6.64,8.05
Splitter,6.82,7.08,11.39


In [21]:
movimiento_total["reduccion_%"] = (
    (
        movimiento_total["No Altitude"] -
        movimiento_total["Extreme Altitude"]
    )
    / movimiento_total["No Altitude"]
    * 100
).round(1)

movimiento_total

altitude_category,Extreme Altitude,Medium Altitude,No Altitude,reduccion_%
AutoPitchType,,,,
Changeup,11.64,12.55,16.28,28.5
Curveball,10.34,11.19,13.58,23.9
Cutter,8.78,9.97,11.39,22.9
Four-Seam,15.19,16.38,19.85,23.5
Sinker,14.53,15.62,19.12,24.0
Slider,6.07,6.64,8.05,24.6
Splitter,6.82,7.08,11.39,40.1


### Revisión de SpinRate

Para verificar que la diferencia de movimiento no se deba únicamente a que los lanzamientos tienen distinto spin, comparo el SpinRate promedio entre niveles de altitud para cada tipo de lanzamiento.

In [22]:
spin_altitud = (
    h1_principal
    .groupby(
        ["AutoPitchType", "altitude_category"],
        observed=True
    )["SpinRate"]
    .mean()
    .unstack()
    .round(0)
)

spin_altitud

altitude_category,Extreme Altitude,Medium Altitude,No Altitude
AutoPitchType,,,
Changeup,1834.0,1799.0,1802.0
Curveball,2545.0,2534.0,2519.0
Cutter,2355.0,2397.0,2426.0
Four-Seam,2321.0,2336.0,2320.0
Sinker,2267.0,2257.0,2245.0
Slider,2433.0,2448.0,2398.0
Splitter,1165.0,1073.0,1103.0


### Conclusión H1

Los resultados obtenidos apoyan la hipótesis 1: a mayor altitud, los lanzamientos tienden a presentar menor movimiento inducido por spin.

Al comparar todos los lanzamientos de forma general, el movimiento vertical inducido promedio disminuyó conforme aumentó la altitud. 

Después, al separar el análisis por tipo de lanzamiento, el patrón se mantuvo en los siete tipos principales analizados.

Calculamos también la magnitud total de movimiento y en todos los casos obervamos: `No Altitude > Medium Altitude > Extreme Altitude`

La reducción entre `No Altitude` y `Extreme Altitude` fue aproximadamente de:

- 28.5% en Changeup
- 23.9% en Curveball
- 22.9% en Cutter
- 23.5% en Four-Seam
- 24.0% en Sinker
- 24.6% en Slider
- 40.1% en Splitter

También se revisó el `SpinRate` promedio por tipo de lanzamiento y nivel de altitud. 
* No se observó una disminución general del spin en `Extreme Altitude`.
* En varios tipos de lanzamiento el SpinRate fue prácticamente igual o incluso ligeramente mayor que en `No Altitude`.

Esto sugiere que la reducción del movimiento observada en altitud no se explica simplemente porque los lanzamientos tengan menos spin.

En conclusión, el patrón es consistente entre los distintos tipos de lanzamiento y respalda la hipótesis.

# H3. Los groundball pitchers son más valiosos en altitud

## Idea del análisis

Para esta hipótesis, primero vamos a calcular qué proporción de las bolas puestas en juego contra cada pitcher son groundballs.

Entonces nos queda: `GB% = groundballs / bolas en juego`

Después dividiremos a los pitchers en dos grupos:

- `High GB`: pitchers con GB% por encima de la mediana.
- `Low GB`: pitchers con GB% por debajo de la mediana.

Luego se comparará su desempeño en distintos niveles de altitud.

In [23]:
df["hit_type"].value_counts(dropna=False)

hit_type
None          450965
GroundBall     63890
FlyBall        50371
LineDrive      37801
Popup          29694
Bunt            2281
Name: count, dtype: int64

In [24]:
df["is_ball_in_play"].value_counts(dropna=False)

is_ball_in_play
0.0    519054
1.0    115948
Name: count, dtype: int64

## Revisión de las bolas en juego

Antes de calcular el porcentaje de groundballs, se revisa cómo se relacionan `hit_type` con `is_ball_in_play`.

* `hit_type`: describe el tipo de contacto registrado
*  `is_ball_in_play`: indica si el lanzamiento terminó con una pelota puesta oficialmente en juego

Como ambas variables no tienen la misma cantidad de observaciones, primero se verifica su relación antes de definir el denominador de `GB%`.

In [25]:
pd.crosstab(
    df["hit_type"],
    df["is_ball_in_play"],
    dropna=False
)

is_ball_in_play,0.0,1.0
hit_type,,
Bunt,0,2281
FlyBall,26444,23927
GroundBall,14787,49103
LineDrive,13204,24597
Popup,20203,9491
NaN,444416,6549


In [26]:
df.groupby("hit_type", dropna=False)[
    ["is_ball_in_play", "is_batted"]
].mean().round(3)

,is_ball_in_play,is_batted
hit_type,,
Bunt,1.000,1.000
FlyBall,0.475,1.000
GroundBall,0.769,1.000
LineDrive,0.651,1.000
Popup,0.320,1.000
NaN,0.015,0.092


## Cálculo de GB% por pitcher

Para identificar qué pitchers generan más groundballs, se calcula el porcentaje de contactos clasificados como `GroundBall` de la siguiente manera:

`GB% = GroundBalls / contactos con hit_type conocido`

Se utilizan únicamente contactos donde `hit_type` está identificado, ya que los valores faltantes no permiten saber si fueron groundballs u otro tipo de contacto.

In [27]:
contactos = df[df["hit_type"].notna()].copy()

contactos["is_groundball"] = (
    contactos["hit_type"] == "GroundBall"
).astype(int)

gb_pitcher = (
    contactos
    .groupby("pitcher_anon_id")
    .agg(
        contactos=("hit_type", "size"),
        groundballs=("is_groundball", "sum"),
        GB_pct=("is_groundball", "mean")
    )
)

gb_pitcher["GB_pct"] = gb_pitcher["GB_pct"] * 100

gb_pitcher.head()

,contactos,groundballs,GB_pct
pitcher_anon_id,,,
pitcher_00001,4,2,50.000000
pitcher_00002,152,64,42.105263
pitcher_00003,29,5,17.241379
pitcher_00004,30,10,33.333333
pitcher_00005,31,9,29.032258


In [28]:
gb_pitcher["contactos"].describe()

count    1127.000000
mean      162.479148
std       192.014582
min         1.000000
25%        23.000000
50%        93.000000
75%       224.500000
max      1136.000000
Name: contactos, dtype: float64

In [29]:
gb_pitcher.shape

(1127, 3)

## Selección de pitchers

Algunos pitchers tienen muy pocos contactos registrados, por lo que su GB% no es tan representativo.

Para evitar clasificaciones basadas en muestras demasiado pequeñas, se consideran únicamente pitchers con al menos 50 contactos clasificados.

Después, los pitchers se dividen en dos grupos usando la mediana del GB%:

- `High GB`: GB% mayor o igual a la mediana.
- `Low GB`: GB% menor a la mediana.

In [30]:
gb_pitcher_filtrado = gb_pitcher[
    gb_pitcher["contactos"] >= 50
].copy()

gb_pitcher_filtrado.shape

(730, 3)

In [31]:
mediana_gb = gb_pitcher_filtrado["GB_pct"].median()

mediana_gb

34.39203354297693

In [32]:
gb_pitcher_filtrado["GB_group"] = (
    gb_pitcher_filtrado["GB_pct"] >= mediana_gb
).map({
    True: "High GB",
    False: "Low GB"
})

gb_pitcher_filtrado["GB_group"].value_counts()

GB_group
High GB    365
Low GB     365
Name: count, dtype: int64

## Comparación de valor por altitud

Para revisar si los pitchers con mayor GB% son más valiosos en altitud, se comparan dos medidas de daño permitido:

- `HR%`: porcentaje de contactos que terminan en home run.
- `Runs per 100 contacts`: carreras anotadas por cada 100 contactos.

En ambos casos, un valor menor representa un mejor resultado para el pitcher.

La comparación se hace entre los grupos `High GB` y `Low GB` dentro de cada nivel de altitud.

In [35]:
grupo_pitcher = gb_pitcher_filtrado["GB_group"]

h3 = df[df["pitcher_anon_id"].isin(grupo_pitcher.index)].copy()

h3["GB_group"] = h3["pitcher_anon_id"].map(grupo_pitcher)

In [36]:
h3_contactos = h3[h3["is_batted"] == 1].copy()

In [37]:
resultado_h3 = (
    h3_contactos
    .groupby(
        ["altitude_category", "GB_group"],
        observed=True
    )
    .agg(
        contactos=("PitchUID", "size"),
        HR_rate=("home_run", "mean"),
        carreras_promedio=("RunsScored", "mean")
    )
)

resultado_h3["HR_pct"] = resultado_h3["HR_rate"] * 100
resultado_h3["Runs_per_100_contacts"] = (
    resultado_h3["carreras_promedio"] * 100
)

resultado_h3[
    ["contactos", "HR_pct", "Runs_per_100_contacts"]
].round(2)

contactos  HR_pct  Runs_per_100_contacts
altitude_category GB_group                                          
Extreme Altitude  High GB       33375    2.10                  11.99
                  Low GB        28696    2.40                  11.62
Medium Altitude   High GB       28116    2.01                  10.86
                  Low GB        25506    2.36                  10.90
No Altitude       High GB       50248    1.71                   8.42
                  Low GB        50527    1.94                   7.85

### Primer resultado

Los pitchers del grupo `High GB` permiten un menor porcentaje de home runs que los del grupo `Low GB` en los tres niveles de altitud.

En `Extreme Altitude`, el HR% es de 2.10% para `High GB` y 2.40% para `Low GB`.

Sin embargo, esta ventaja no aparece en las carreras permitidas por cada 100 contactos: en `Extreme Altitude`, los pitchers `High GB` presentan 11.99 carreras por 100 contactos frente a 11.62 de `Low GB`.

Por lo tanto, hasta este punto hay evidencia de una ventaja de los pitchers con mayor GB% para evitar home runs, pero no de una reducción general de carreras permitidas.

## Diferencia entre High GB y Low GB

Para evaluar si la ventaja de generar groundballs aumenta con la altitud, calculo la diferencia entre los grupos `High GB` y `Low GB` dentro de cada nivel de altitud.

In [38]:
comparacion_h3 = (
    resultado_h3[
        ["HR_pct", "Runs_per_100_contacts"]
    ]
    .unstack("GB_group")
)

comparacion_h3["Diferencia_HR_pct"] = (
    comparacion_h3[("HR_pct", "Low GB")]
    - comparacion_h3[("HR_pct", "High GB")]
)

comparacion_h3["Diferencia_runs"] = (
    comparacion_h3[("Runs_per_100_contacts", "Low GB")]
    - comparacion_h3[("Runs_per_100_contacts", "High GB")]
)

comparacion_h3.round(2)

HR_pct        Runs_per_100_contacts         \
GB_group          High GB Low GB               High GB Low GB   
altitude_category                                               
Extreme Altitude     2.10   2.40                 11.99  11.62   
Medium Altitude      2.01   2.36                 10.86  10.90   
No Altitude          1.71   1.94                  8.42   7.85   

                  Diferencia_HR_pct Diferencia_runs  
GB_group                                             
altitude_category                                    
Extreme Altitude               0.30           -0.36  
Medium Altitude                0.35            0.05  
No Altitude                    0.23           -0.58

### Comparación de la ventaja por altitud

La diferencia de HR% entre `High GB` y `Low GB` es mayor en los niveles de altitud que en `No Altitude`.

- `No Altitude`: 0.23 puntos porcentuales
- `Medium Altitude`: 0.35 puntos porcentuales
- `Extreme Altitude`: 0.30 puntos porcentuales

Esto sugiere que los pitchers con mayor GB% tienen una ventaja algo mayor para evitar home runs en altitud.

Sin embargo, esta misma ventaja no aparece en carreras permitidas. En `Extreme Altitude`, los pitchers `High GB` incluso presentan ligeramente más carreras por 100 contactos que los `Low GB`.

Por ahora, H3 parece tener apoyo únicamente en la prevención de home runs, no en el daño total permitido.

## Extra-base hits

También se compara la frecuencia de extra-base hits (`XBH`), definidos como dobles, triples o home runs.

Esta medida permite revisar si los pitchers con mayor GB% reducen contactos de alto impacto en general, y no únicamente home runs.

In [ ]:
h3_contactos["extra_base_hit"] = (
    (h3_contactos["double"] == 1) |
    (h3_contactos["triple"] == 1) |
    (h3_contactos["home_run"] == 1)
).astype(int)

xbh_h3 = (
    h3_contactos
    .groupby(
        ["altitude_category", "GB_group"],
        observed=True
    )
    .agg(
        contactos=("PitchUID", "size"),
        XBH_rate=("extra_base_hit", "mean")
    )
)

xbh_h3["XBH_pct"] = xbh_h3["XBH_rate"] * 100


xbh_h3[
    ["contactos", "XBH_pct"]
].round(2)

contactos  XBH_pct
altitude_category GB_group                    
Extreme Altitude  High GB       33375     6.46
                  Low GB        28696     6.75
Medium Altitude   High GB       28116     6.18
                  Low GB        25506     6.43
No Altitude       High GB       50248     5.04
                  Low GB        50527     4.94

### Conclusión H3

Para revisar H3, los pitchers se dividieron en dos grupos según su porcentaje de groundballs (`GB%`):

- `High GB`: pitchers con GB% igual o mayor a la mediana (34.39%).
- `Low GB`: pitchers con GB% menor a la mediana.

Solo se consideraron pitchers con al menos 50 contactos registrados, quedando 730 pitchers: 365 en cada grupo.

Los pitchers `High GB` mostraron un menor porcentaje de home runs en los tres niveles de altitud. La diferencia frente a `Low GB` fue:

- `No Altitude`: 0.23 puntos porcentuales menos de HR.
- `Medium Altitude`: 0.35 puntos porcentuales menos.
- `Extreme Altitude`: 0.30 puntos porcentuales menos.

La ventaja para evitar home runs fue ligeramente mayor en los niveles de altitud que en `No Altitude`.

Hay una pequeña ventaja de `High GB` en altitud:

- `Extreme Altitude`: 6.46% vs 6.75%.
- `Medium Altitude`: 6.18% vs 6.43%.

NOTA: En `No Altitude` esta ventaja desaparece (5.04% vs 4.90%).

* Las carreras permitidas por cada 100 contactos no muestran una ventaja clara para los pitchers `High GB`. En `Extreme Altitude`, incluso presentan un valor ligeramente mayor: 11.99 frente a 11.62.

**Conclusión:** H3 no se afirma por completo, solo parcialmente. 

Los pitchers con mayor GB% parecen tener una pequeña ventaja en altitud para limitar home runs y extra-base hits, pero esta ventaja no se significa una reducción clara de las carreras permitidas. 

Por lo tanto, con este análisis no se puede concluir que los groundball pitchers sean más valiosos en altitud.

In [42]:
df.head(10)

,year,PitchUID,game_anon_id,pitcher_anon_id,batter_anon_id,catcher_anon_id,PitcherThrows,BatterSide,altitude_category,AutoPitchType,...,Top/Bottom,Outs,Balls,Strikes,count,PlateLocHeight,PlateLocSide,in_strike_zone,outside_strike_zone,swung_outside_strike_zone
0,2024,d6a82560-f939-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Sinker,...,Top,0.0,0.0,0.0,0-0,4.19842,-1.06959,0,1.0,0.0
1,2024,df15bd20-f939-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Sinker,...,Top,0.0,1.0,0.0,1-0,1.75154,-0.20261,1,0.0,0.0
2,2024,e965d080-f939-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Cutter,...,Top,0.0,1.0,1.0,1-1,2.05249,1.41929,0,1.0,0.0
3,2024,f1128940-f939-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Cutter,...,Top,0.0,2.0,1.0,2-1,3.31819,-0.03459,0,1.0,1.0
4,2024,ffee8d60-f939-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Curveball,...,Top,0.0,2.0,2.0,2-2,1.43131,0.53384,0,1.0,0.0
5,2024,0a1769b0-f93a-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00272,catcher_00123,Left,Right,No Altitude,Cutter,...,Top,0.0,3.0,2.0,3-2,1.53276,-0.86929,0,1.0,0.0
6,2024,1f92ae30-f93a-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00808,catcher_00123,Left,Right,No Altitude,Sinker,...,Top,1.0,0.0,0.0,0-0,2.97963,1.15997,0,1.0,0.0
7,2024,2868a4b0-f93a-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00808,catcher_00123,Left,Right,No Altitude,Sinker,...,Top,1.0,1.0,0.0,1-0,2.59802,0.04399,1,0.0,0.0
8,2024,345d6760-f93a-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00808,catcher_00123,Left,Right,No Altitude,Cutter,...,Top,1.0,1.0,1.0,1-1,3.23001,1.12929,0,1.0,1.0
9,2024,474ef3c0-f93a-11ee-8e28-698264036ef0,game_001331,pitcher_01067,batter_00808,catcher_00123,Left,Right,No Altitude,Curveball,...,Top,1.0,1.0,2.0,1-2,2.81374,-0.18737,1,0.0,0.0
